# 1. Import

In [ ]:
# pip install tensorflow==2.10 scikit-learn matplotlib pandas pillow

import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten
from tensorflow.keras.layers import Rescaling, RandomFlip, RandomRotation
from tensorflow.keras import layers, models

from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications import (
    MobileNetV3Large,
    DenseNet121,
    ResNet50,
    EfficientNetB3
)

from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
from datetime import datetime
import random

%matplotlib inline

In [ ]:
# tag: parameters
DATASET_NAME = " "  # papermill จะ inject ให้อัตโนมัติ

In [ ]:
train_path = f"{DATASET_NAME}/train"
valid_path = f"{DATASET_NAME}/valid"
test_path  = f"{DATASET_NAME}/test"

In [ ]:
# กำหนด Hyperparameters
ROTATION_RANGE = 0
HORIZONTAL_FLIP = False
VERTICAL_FLIP = False
ZOOM_RANGE = 0
WIDTH_SHIFT_RANGE = 0
HEIGHT_SHIFT_RANGE = 0
SHEAR_RANGE = 0

BATCH_SIZE = 128
IMAGE_SIZE = 224

In [ ]:
MODEL = 1

MODELS = {
    1: (MobileNetV3Large, "MobileNetV3Large"),
    2: (DenseNet121, "DenseNet121"),
    3: (ResNet50, "ResNet50"),
    4: (EfficientNetB3, "EfficientNetB3")
}

MODEL_CLASS, MODEL_NAME = MODELS[MODEL]


if MODEL == 1:
    from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

elif MODEL == 2:
    from tensorflow.keras.applications.densenet import preprocess_input

elif MODEL == 3:
    from tensorflow.keras.applications.resnet50 import preprocess_input

elif MODEL == 4:
    from tensorflow.keras.applications.efficientnet import preprocess_input

print(f"Using model: {MODEL_NAME}")

In [ ]:
# Run ImageDataGenerator เพื่อเตรียมข้อมูลสำหรับการฝึกโมเดล
train_datagen = ImageDataGenerator(
      width_shift_range = WIDTH_SHIFT_RANGE,
      height_shift_range = HEIGHT_SHIFT_RANGE,
      rotation_range = ROTATION_RANGE,
      horizontal_flip = HORIZONTAL_FLIP,
      vertical_flip = VERTICAL_FLIP,
      zoom_range = ZOOM_RANGE,
      preprocessing_function = preprocess_input
)

train_generator = train_datagen.flow_from_directory(
      directory=train_path,
      target_size=(IMAGE_SIZE, IMAGE_SIZE),
      batch_size=BATCH_SIZE,
      class_mode='categorical'
)

test_datagen = ImageDataGenerator(
     preprocessing_function = preprocess_input
)

test_generator = test_datagen.flow_from_directory(
      directory=test_path,
      target_size=(IMAGE_SIZE, IMAGE_SIZE),
      batch_size=BATCH_SIZE,
      shuffle=False,
      class_mode='categorical'
)

valid_datagen = ImageDataGenerator(
     preprocessing_function = preprocess_input
)

valid_generator = valid_datagen.flow_from_directory(
      directory=valid_path,
      target_size=(IMAGE_SIZE, IMAGE_SIZE),
      batch_size=BATCH_SIZE,
      shuffle=False,
      class_mode='categorical')

In [ ]:
# ดึงข้อมูล Class
class_names = train_generator.class_indices
class_names

In [ ]:
# ตัวแปร CLASS_SIZE เก็บจำนวนคลาสทั้งหมดในชุดข้อมูลฝึกโมเดล
CLASS_SIZE = len(class_names)
CLASS_SIZE

In [ ]:
# ตั้งค่า Seed
SEED = 42
# SEED = np.random.randint(1, 100) # สุ่ม SEED ระหว่าง 1 ถึง 100
print(f"Random Seed: {SEED}")

os.environ['PYTHONHASHSEED']=str(SEED)
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

inputs = tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="input_layer")

In [ ]:
# เรียกใช้โมเดล

base_model = MODEL_CLASS(
    include_top=False,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    weights="imagenet"
)

base_model.trainable = False

In [ ]:
x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

outputs = layers.Dense(CLASS_SIZE, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# Compile
model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(0.0001),
              metrics=["accuracy"])

In [ ]:
# ตั้งค่า Early Stopping
custom_early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    min_delta=0.001,
    restore_best_weights=True # บันทึกค่าที่ดีที่สุด
)

In [ ]:
# เริ่ม Train
start = datetime.now()

history = model.fit(train_generator,
                    validation_data=valid_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=100, # จำนวน Epoch
                    callbacks=[custom_early_stopping])

print('Execution Time: ', datetime.now()-start)

In [ ]:
model.summary()

In [ ]:
# Save เป็น Keras format
model.save(f"EfficientNetB3_{DATASET_NAME}.keras")

# Save เป็น HDF5 (.h5)
model.save(f"EfficientNetB3_{DATASET_NAME}.h5")

In [ ]:
# ประเมินผลการฝึกโมเดลด้วยชุดข้อมูลฝึก (Train)
train_loss, train_acc = model.evaluate(
    train_generator,
    steps=len(train_generator),
    # verbose=0
)
print('Train loss:', train_loss)
print('Train accuracy:', train_acc)

In [ ]:
# ประเมินผลการฝึกโมเดลด้วยชุดข้อมูลทดสอบ (Valid)
valid_loss, valid_acc = model.evaluate(
    valid_generator,
    steps=len(valid_generator),
    # verbose=0
)
print('Valid loss:', valid_loss)
print('Valid accuracy:', valid_acc)

In [ ]:
# ประเมินผลการฝึกโมเดลด้วยชุดข้อมูลทดสอบ (Test)
test_loss, test_acc = model.evaluate(
    test_generator,
    steps=len(test_generator),
    # verbose=0
)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

In [ ]:
# ส่วนการทำ Confusion Matrix และ Report
import sklearn as scikit_learn
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_recall_fscore_support
import numpy as np

# Train Set Confusion Matrix
train_true=train_generator.classes[train_generator.index_array]
train_pred_raw = model.predict(train_generator)
train_pred = np.argmax(train_pred_raw, axis=1)

cm = confusion_matrix(train_true, train_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(10,10))
disp.plot(ax=ax,cmap=plt.cm.Blues)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right') # ทำให้ชื่อ class ในแนวแกน x เอียง เพื่อให้อ่านง่าย
plt.show()

res = []
for l in range(CLASS_SIZE):
    pres,recall,_,_ = precision_recall_fscore_support(np.array(train_true)==l, np.array(train_pred)==l, pos_label=True, average=None)
    res.append([l, recall[0], recall[1]])

pd.DataFrame(res,columns = ['class', 'specificity', 'sensitivity'])

In [ ]:
# Valid Set Confusion Matrix
valid_true=valid_generator.classes[valid_generator.index_array]
valid_pred_raw = model.predict(valid_generator)
valid_pred = np.argmax(valid_pred_raw, axis=1)

cm = confusion_matrix(valid_true, valid_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(10,10))
disp.plot(ax=ax,cmap=plt.cm.Blues)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right') # ทำให้ชื่อ class ในแนวแกน x เอียง เพื่อให้อ่านง่าย
plt.show()

res = []
for l in range(CLASS_SIZE):
    pres,recall,_,_ = precision_recall_fscore_support(np.array(valid_true)==l, np.array(valid_pred)==l, pos_label=True, average=None)
    res.append([l, recall[0], recall[1]])

pd.DataFrame(res, columns = ['class', 'specificity', 'sensitivity'])

In [ ]:
# Test Set Confusion Matrix
test_true = test_generator.classes[test_generator.index_array]
test_pred_raw = model.predict(test_generator)
test_pred = np.argmax(test_pred_raw, axis=1)

cm = confusion_matrix(test_true, test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(10,10))
disp.plot(ax=ax, cmap=plt.cm.Blues)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.show()

res = []
for l in range(CLASS_SIZE):
    pres,recall,_,_ = precision_recall_fscore_support(np.array(test_true)==l, np.array(test_pred)==l, pos_label=True, average=None)
    res.append([l, recall[0], recall[1]])

pd.DataFrame(res, columns = ['class', 'specificity', 'sensitivity'])

In [ ]:
# Report for Train Set
print(classification_report(train_true, train_pred, target_names=class_names, digits=4))

In [ ]:
# Report for Valid Set
print(classification_report(valid_true, valid_pred, target_names=class_names, digits=4))

In [ ]:
# Report for Test Set
print(classification_report(test_true, test_pred, target_names=class_names, digits=4))

In [ ]:
# Plot Graph
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, 'r', label='Training Accuracy')
plt.plot(epochs_range, val_acc, 'b', label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, 'r', label='Training Loss')
plt.plot(epochs_range, val_loss, 'b', label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()